Objetivos: Hacer decodificador ASTERIX para CAT048 y CAT021-v2.1  Hay que exportar los resultados a CSV en el formato especificado (eligiendo donde guardarlo).
Objetivos secundarios:
-          Altitud corregida con QNH (cuando haya columna extra)
-          Todos los blancos tienen posición en latitud, longitud y h en coordenadas WGS84 (Columnas extra para cada valor)
-          Los datos se podrán filtrar
-           La simulación tendrá la capacidad de diferenciar el blanco dependiendo del sistema que lo ha detectado (Radar/ADS-B)
-           Desarrollar el SW lo más eficiente posible
NOTAS:
-          No se deben decodificar los DI del CAT048 marcados en rojo.
-          Descartar aeronaves fuera del filtro geográfico (1r pdf).
-          REVISAR EL RESTO EN EL 1r PDF
 
1.      Uno de los primeros pasos es investigar como abrir un fichero binario:
Para ello la mejor forma de hacerlo es utilizar un “Hex Editor”. Este es una extensión instalable de VScode. De esta manera podemos trabajar con el código a la vez que tenemos los archivos .ast abiertos.
Recordamos que un octeto (1 byte) son dos dígitos en hexadecimal.

CLASIFICACION DE LA INFORMACIÓN:
Data categories (up to 256 categories): 
El fichero esta formado por data block. Los tres primeros bytes de cada datablock son la cabecera del datablock. El primero nos dice la categoría y los dos segundos la length del bloque. La payload son 3 octetos.

Entonces tenemos dos tipos de categorias, la 48 y la 21. En el fichero combinado encontraremos ambas.

Un datablock puede tener uno o varios Records. Cada Record empieza por un FSPEC y luego vienen los data fields. Los datafields se envian en order de FRN creciente. La longitud de cada Record se deduce al ir consumiendo bytes segun indica el FSPEC + el formato de cada data field. El FSPEC es un campo de longitud variable que indica cuantos campos vienen en el record, una lista de los FRNs presentes. Luego hay que traducir los FRNs segun la table. Estos datos se miran directamente de los documentos que hacen de diccionario. Ademas hay longitudes variables que deberemos estudiar.


In [2]:
from pathlib import Path
from datetime import timedelta

def hexdump(b: bytes, width: int = 16, start_offset: int = 0) -> str:
    """Devuelve un hexdump estilo visor hex (offset + bytes)."""
    lines = []
    for i in range(0, len(b), width):
        chunk = b[i:i+width]
        hex_part = " ".join(f"{x:02X}" for x in chunk)
        lines.append(f"{start_offset+i:08X}  {hex_part}")
    return "\n".join(lines)

def read_datablock(raw: bytes, offset: int = 0):
    """Lee un datablock ASTERIX: CAT(1) + LEN(2) + payload(LEN-3)."""
    cat = raw[offset]
    length = int.from_bytes(raw[offset+1:offset+3], "big")
    payload = raw[offset+3:offset+length]
    return cat, length, payload

def read_fspec(payload: bytes, pos: int = 0):
    """
    Lee FSPEC: uno o más octetos.
    FX = bit menos significativo (LSB) de cada octeto:
      FX=1 -> hay otro octeto FSPEC
      FX=0 -> fin FSPEC
    """
    fspec = []
    while True:
        b = payload[pos]
        fspec.append(b)
        pos += 1
        if (b & 0x01) == 0:  # FX=0
            break
    return fspec, pos

def fspec_to_frns(fspec: list[int]) -> list[int]:
    """
    Convierte FSPEC -> lista de FRNs presentes.
    En cada octeto FSPEC:
      bits 8..2 (más significativos) indican FRN base+1..base+7
      bit 1 (LSB) es FX
    """
    masks = [0x80, 0x40, 0x20, 0x10, 0x08, 0x04, 0x02]
    frns = []
    for n, b in enumerate(fspec):
        base = n * 7
        for i, m in enumerate(masks, start=1):
            if b & m:
                frns.append(base + i)
    return frns

def tod_to_hhmmss(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

In [ ]:
AST_PATH = "raw_data/datos_asterix_adsb.ast"
# Cambia esto a tu ruta (o déjalo si el .ast está en la misma carpeta del notebook)
raw = Path(AST_PATH).read_bytes()

# 1) Primer datablock
cat, length, payload = read_datablock(raw, offset=0)

print("== CABECERA DATABLOCK ==")
print(f"CAT   = {cat} (0x{cat:02X})")
print(f"LEN   = {length} (0x{length:04X}) bytes -> este bloque ocupa bytes [0 .. {length-1}]")
print("\nPrimeros 64 bytes del fichero (para comparar con tu visor hex):")
print(hexdump(raw[:64]))

# 2) FSPEC
fspec, pos = read_fspec(payload, pos=0)
frns = fspec_to_frns(fspec)

print("\n== FSPEC ==")
print("FSPEC bytes =", " ".join(f"{b:02X}" for b in fspec), f"(len={len(fspec)} octetos)")
print("FRNs presentes =", frns)

# 3) Decodificación de ejemplo de los primeros campos que aparecen en ESTE mensaje (CAT021)
#    (Usamos la tabla UAP de CAT021: pág. 16 del PDF de hackathon)
#    FRN1  I021/010 (2)  -> SAC/SIC
#    FRN2  I021/040 (1+) -> Target Report Descriptor (aquí lo leemos con FX)
#    FRN3  I021/161 (2)  -> Track Number
#    FRN4  I021/015 (1)  -> Service Identification
#    FRN7  I021/131 (8)  -> WGS-84 high res (lat/lon)
#    FRN11 I021/080 (3)  -> Target Address (ICAO24)
#    FRN12 I021/073 (3)  -> Time of Message Reception of Position

def read_fx_field(buf: bytes, pos: int) -> tuple[bytes, int]:
    start = pos
    while True:
        b = buf[pos]
        pos += 1
        if (b & 0x01) == 0:  # FX=0
            break
    return buf[start:pos], pos

decoded = {}

# FRN1 I021/010 (2)
decoded["SAC"] = payload[pos]
decoded["SIC"] = payload[pos+1]
pos += 2

# FRN2 I021/040 (1+) -> lo dejamos como bytes (en este mensaje son 3)
trd_raw, pos = read_fx_field(payload, pos)
decoded["I021/040_raw_hex"] = trd_raw.hex().upper()
decoded["I021/040_len"] = len(trd_raw)

# FRN3 I021/161 (2)
decoded["TrackNumber"] = int.from_bytes(payload[pos:pos+2], "big")
pos += 2

# FRN4 I021/015 (1)
decoded["ServiceID"] = payload[pos]
pos += 1

# FRN7 I021/131 (8) -> lat/lon (int32 signed) -> grados
lat_i = int.from_bytes(payload[pos:pos+4], "big", signed=True); pos += 4
lon_i = int.from_bytes(payload[pos:pos+4], "big", signed=True); pos += 4
decoded["lat_raw_int"] = lat_i
decoded["lon_raw_int"] = lon_i
decoded["lat_deg"] = lat_i * 180 / (2**30)   # (escala de CAT021 para high-res)
decoded["lon_deg"] = lon_i * 180 / (2**30)

# FRN11 I021/080 (3) -> ICAO24
decoded["ICAO24_hex"] = payload[pos:pos+3].hex().upper()
pos += 3

# FRN12 I021/073 (3) -> Time of Message Reception of Position (típicamente /128 s)
t = int.from_bytes(payload[pos:pos+3], "big"); pos += 3
decoded["ToD_raw"] = t
decoded["ToD_seconds"] = t / 128
decoded["ToD_hhmmss"] = tod_to_hhmmss(decoded["ToD_seconds"])

print("\n== DECODIFICACIÓN (parcial, ejemplo) ==")
for k, v in decoded.items():
    print(f"{k:16s} = {v}")

print(f"\nPuntero actual dentro del payload: pos={pos} (quedan {len(payload)-pos} bytes por interpretar en este record)")
print("Los siguientes 32 bytes (por si quieres seguir manualmente):")
print(hexdump(payload[pos:pos+32], start_offset=3 + pos))  # +3 por CAT+LEN

NameError: name 'AST_PATH' is not defined

CAT y Length:   Para explicarlo si vemos el inicio del archivo el primer byte es un 15 en hexadecimal que equivale a 21 en decimal. Esto indica que es categoria 21. Luego 00 5A es la longitud que equivale a  los proximos 90  bytes. Con esta logintud sabemos donde acaba para saltar al siguiente datablock. 

FSPEC:  Despues de la length viene el FSPEC. Este hace de índice indicando los data fields que estan presentes en el record. Cada octeto de FSPEC tiene un bit especial (FX) que indica si hay otro octeto de FSPEC después (el bit menos significativo). Si FX=1 → sigue el FSPEC, Si FX=0 → se acaba. En este caso, el último octeto es 04: 0x04 en binario es 00000100 su bit menos significativo (FX) es 0 ⇒ fin del FSPEC. Por esta razón el FSPEC tiene 7 bytes. El FSPEC tiene longitud variable con el FX.

FRNs: Indican los data fields presentes, luego dentro del record los campose se envían en orden FRN creciente. Cada byte/octeto del FSPEC indica 7 FRNs, ya que el octavo bit es el FX. 1 es que esta presente y 0 que no. Si estamos en el octeto nº1 del FSPEC (el primero), entonces:

bit 8 → FRN 1
bit 7 → FRN 2
bit 6 → FRN 3
bit 5 → FRN 4
bit 4 → FRN 5
bit 3 → FRN 6
bit 2 → FRN 7
bit 1 → FX (NO es un FRN)

Luego ya vienen los data source idenfications y parametros mas concretos que dependen de la categoria en la que estemos. Esto lo esta haciendo HUGO, así que lo dejo hasta aquí que es la clasificacion más basica. 